# Experiments with boosting models

After modeling and error analysis, we explore additional experiments to improve predictions without significantly increasing computational cost. We focus on a boosting model that can capture numerical relationships CatBoost may miss, produce less-correlated errors, train efficiently, and handle high-dimensional feature spaces. 

These considerations lead us to LightGBM. We train LightGBM, compare its performance with CatBoost, and then evaluate whether combining their predictions can further improve our results.

## LightGBM and CatBoost 

This notebook compares LightGBM with the CatBoost model, then chooses a validation-based blend. It uses the same cleaned-enriched data and chronological split:

- training: January 2021 to March 2024
- validation: April 2024 to March 2025

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from time import perf_counter


TARGET = "MONTHLY_RENT"
train = pd.read_csv("derived/train_cleaned_enriched.csv")
test = pd.read_csv("derived/test_cleaned_enriched.csv")
print("Train shape:", train.shape)
print("Test shape: ", test.shape)

Train shape: (149841, 21)
Test shape:  (50000, 20)


In [2]:
# Match Models_main.ipynb exactly: hold out the latest 12 months.
approval_dates = pd.to_datetime(train["RENT_APPROVAL_DATE"])
cutoff = approval_dates.max().to_period("M").start_time - pd.DateOffset(months=12)
is_validation = approval_dates > cutoff
training_data = train.loc[~is_validation].copy()
validation_data = train.loc[is_validation].copy()
print(f"Cutoff month: {cutoff:%Y-%m}")
print(f"Training rows: {len(training_data):,}; validation rows: {len(validation_data):,}")

def split_xy(data):
    X = data.drop(columns=[TARGET], errors="ignore").copy()
    y = data[TARGET].copy() if TARGET in data.columns else None
    return X, y

X_fit_raw, y_fit = split_xy(training_data)
X_valid_raw, y_valid = split_xy(validation_data)
X_all_raw, y_all = split_xy(train)
X_test_raw, _ = split_xy(test)

Cutoff month: 2024-03
Training rows: 114,683; validation rows: 35,158


## Shared feature schema

CatBoost uses categorical columns natively. 

LightGBM receives a matrix made with the same numeric columns, one-hot encoded low-cardinality categories, and target-encoded `LOCATION_KEY`. Its preprocessor is fitted only on the training partition for validation, so validation targets cannot leak into features.

In [3]:
NUMERIC_FEATURES = [
    "FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "HDB_LATITUDE",
    "HDB_LONGITUDE", "HDB_YEAR_COMPLETED", "NEAREST_MRT_KM",
    "NEAREST_SCHOOL_KM", "NEAREST_MALL_KM", "APPROVAL_YEAR",
    "APPROVAL_MONTH", "MONTH_INDEX", "BUILDING_AGE",
]
ONEHOT_FEATURES = ["TOWN", "FLAT_TYPE", "FLAT_MODEL", "HDB_PLANNING_AREA"]
LOCATION_FEATURES = ["LOCATION_KEY"]
CATBOOST_ALL_FEATURES = NUMERIC_FEATURES + ONEHOT_FEATURES + LOCATION_FEATURES
CATBOOST_CATEGORICAL = ONEHOT_FEATURES + LOCATION_FEATURES

def make_lgbm_preprocessor():
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
    ])
    categorical = Pipeline([
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    location = Pipeline([
        ("encode", TargetEncoder(target_type="continuous", smooth="auto",
                                  cv=5, shuffle=True, random_state=42)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC_FEATURES),
        ("categorical", categorical, ONEHOT_FEATURES),
        ("location", location, LOCATION_FEATURES),
    ], remainder="drop", sparse_threshold=0)

preprocessor = make_lgbm_preprocessor()
X_fit_encoded = preprocessor.fit_transform(X_fit_raw, y_fit)
X_valid_encoded = preprocessor.transform(X_valid_raw)
print("Encoded training shape: ", X_fit_encoded.shape)
print("Encoded validation shape:", X_valid_encoded.shape)

Encoded training shape:  (114683, 95)
Encoded validation shape: (35158, 95)


/Users/apple/Documents/modules/CS5242/cs5242/lib/python3.14/site-packages/sklearn/preprocessing/_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


In [4]:
def make_catboost_features(data):
    X = data[CATBOOST_ALL_FEATURES].copy()
    for column in CATBOOST_CATEGORICAL:
        X[column] = X[column].fillna("<MISSING>").astype(str)
    for column in NUMERIC_FEATURES:
        X[column] = pd.to_numeric(X[column], errors="coerce")
        X[column] = X[column].fillna(X[column].median())
    return X

X_cb_fit = make_catboost_features(training_data)
X_cb_valid = make_catboost_features(validation_data)
X_cb_all = make_catboost_features(train)
X_cb_test = make_catboost_features(test)

# Same CatBoost settings as Models_main.ipynb.
CB_PARAMS = dict(loss_function="RMSE", eval_metric="RMSE", depth=6,
                  learning_rate=0.05, l2_leaf_reg=5, random_seed=42,
                  thread_count=4, allow_writing_files=False)
catboost_validation_model = CatBoostRegressor(iterations=2000, **CB_PARAMS)

start = perf_counter()
catboost_validation_model.fit(
    X_cb_fit, y_fit, cat_features=CATBOOST_CATEGORICAL,
    eval_set=(X_cb_valid, y_valid), early_stopping_rounds=100,
    use_best_model=True, verbose=200)
cb_valid_pred = catboost_validation_model.predict(X_cb_valid)

catboost_time = perf_counter() - start
print(f"CatBoost training time: {catboost_time:.2f} seconds")

cb_iterations = catboost_validation_model.tree_count_
print("Selected CatBoost trees:", cb_iterations)


0:	learn: 713.6667349	test: 823.7971396	best: 823.7971396 (0)	total: 94ms	remaining: 3m 7s
200:	learn: 480.6078977	test: 480.2013346	best: 480.2013346 (200)	total: 3.67s	remaining: 32.8s
400:	learn: 476.7296341	test: 476.2264313	best: 476.2206885 (391)	total: 7.12s	remaining: 28.4s
600:	learn: 474.0167548	test: 474.3569810	best: 474.3569810 (600)	total: 10.8s	remaining: 25.1s
800:	learn: 471.7140243	test: 473.3727622	best: 473.3727622 (800)	total: 14.5s	remaining: 21.6s
1000:	learn: 469.9135725	test: 472.8689593	best: 472.8663302 (999)	total: 18.2s	remaining: 18.2s
1200:	learn: 468.1532489	test: 472.2552763	best: 472.2552763 (1200)	total: 22s	remaining: 14.6s
1400:	learn: 466.6862347	test: 471.9540223	best: 471.9540223 (1400)	total: 25.8s	remaining: 11s
1600:	learn: 465.2165645	test: 471.7173115	best: 471.7173115 (1600)	total: 29.7s	remaining: 7.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 471.701336
bestIteration = 1615

Shrink model to first 1616 iterations.


In [5]:
lightgbm_model = LGBMRegressor(
    objective="regression", n_estimators=3000, learning_rate=0.03,
    num_leaves=31, min_child_samples=30, subsample=0.8,
    colsample_bytree=0.8, reg_lambda=1.0, random_state=42,
    n_jobs=-1, verbosity=-1)

start = perf_counter()
lightgbm_model.fit(
    X_fit_encoded, y_fit, eval_set=[(X_valid_encoded, y_valid)],
    eval_metric="rmse", callbacks=[early_stopping(100), log_evaluation(200)])
lgb_valid_pred = lightgbm_model.predict(
    X_valid_encoded, num_iteration=lightgbm_model.best_iteration_)

lgbm_time = perf_counter() - start
print(f"LightGBM training time: {catboost_time:.2f} seconds")

def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

print("CatBoost validation RMSE: ", rmse(y_valid, cb_valid_pred))
print("LightGBM validation RMSE: ", rmse(y_valid, lgb_valid_pred))
print("LightGBM best iteration: ", lightgbm_model.best_iteration_)

Training until validation scores don't improve for 100 rounds


/Users/apple/Documents/modules/CS5242/cs5242/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's rmse: 475.787	valid_0's l2: 226373
[400]	valid_0's rmse: 473.08	valid_0's l2: 223804
[600]	valid_0's rmse: 472.472	valid_0's l2: 223230
[800]	valid_0's rmse: 472.117	valid_0's l2: 222894
[1000]	valid_0's rmse: 471.973	valid_0's l2: 222758
Early stopping, best iteration is:
[976]	valid_0's rmse: 471.937	valid_0's l2: 222725
LightGBM training time: 32.38 seconds
CatBoost validation RMSE:  471.7013360114649
LightGBM validation RMSE:  471.9374631593723
LightGBM best iteration:  976


CatBoost veryy slightly outperforms LightGBM on the validation set, achieving an RMSE of 471.70 compared with LightGBM’s 471.94. However, their very similar performance suggests that LightGBM may still provide complementary predictions for blending.

### Why a blend of both may work better

A blend can beat either model when their errors are not perfectly correlated. CatBoost and LightGBM both use boosted trees, but they construct useful partitions differently: CatBoost's ordered categorical statistics are useful for categories such as `LOCATION_KEY`, while LightGBM learns from a dense numeric/one-hot/target-encoded matrix and can make different split choices.

The weighted average can keep CatBoost's category-sensitive predictions while reducing some individual errors from either model. This is not guaranteed; the validation RMSE below is the evidence for this dataset. The weight is selected only from the shared time-based validation rows and then held fixed for the final test prediction.

In [6]:
best_blend = {"rmse": float("inf"), "catboost_weight": None, "prediction": None}
for cat_weight in np.linspace(0.0, 1.0, 21):
    blend_pred = cat_weight * cb_valid_pred + (1.0 - cat_weight) * lgb_valid_pred
    blend_rmse = rmse(y_valid, blend_pred)
    if blend_rmse < best_blend["rmse"]:
        best_blend = {"rmse": blend_rmse, "catboost_weight": float(cat_weight),
                      "prediction": blend_pred}

comparison = pd.DataFrame({
    "model": ["CatBoost", "LightGBM", "Blend"],
    "validation_RMSE": [rmse(y_valid, cb_valid_pred),
                          rmse(y_valid, lgb_valid_pred), best_blend["rmse"]]
})
display(comparison.sort_values("validation_RMSE").reset_index(drop=True))
print("Selected CatBoost weight:", best_blend["catboost_weight"])
print("Selected LightGBM weight: ", 1.0 - best_blend["catboost_weight"])

,model,validation_RMSE
0,Blend,470.330585
1,CatBoost,471.701336
2,LightGBM,471.937463


Selected CatBoost weight: 0.5
Selected LightGBM weight:  0.5


In [7]:
# Refit both models on all labeled rows, using validation-selected tree counts and weight.
preprocessor_final = make_lgbm_preprocessor()
X_all_encoded = preprocessor_final.fit_transform(X_all_raw, y_all)
X_test_encoded = preprocessor_final.transform(X_test_raw)

final_catboost = CatBoostRegressor(iterations=cb_iterations, **CB_PARAMS)
final_catboost.fit(X_cb_all, y_all, cat_features=CATBOOST_CATEGORICAL, verbose=200)

final_lightgbm = LGBMRegressor(
    objective="regression", n_estimators=lightgbm_model.best_iteration_,
    learning_rate=0.03, num_leaves=31, min_child_samples=30,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=-1)
final_lightgbm.fit(X_all_encoded, y_all)

cat_weight = best_blend["catboost_weight"]
lgb_weight = 1.0 - cat_weight
cat_test_pred = final_catboost.predict(X_cb_test)
lgb_test_pred = final_lightgbm.predict(X_test_encoded)
final_test_pred = cat_weight * cat_test_pred + lgb_weight * lgb_test_pred

submission = pd.DataFrame({"Id": np.arange(len(final_test_pred)),
                           "Predicted": final_test_pred})
assert len(submission) == len(test)
assert submission["Predicted"].notna().all()
submission.to_csv("predictions/catboost_lightgbm_blend_submission.csv", index=False)
print("Saved predictions/catboost_lightgbm_blend_submission.csv")
display(submission.head())

/Users/apple/Documents/modules/CS5242/cs5242/lib/python3.14/site-packages/sklearn/preprocessing/_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


0:	learn: 730.7584868	total: 36.5ms	remaining: 58.9s
200:	learn: 479.3314282	total: 4.36s	remaining: 30.7s
400:	learn: 475.7494539	total: 8.98s	remaining: 27.2s
600:	learn: 472.8100403	total: 13.5s	remaining: 22.8s
800:	learn: 470.8556164	total: 18s	remaining: 18.3s
1000:	learn: 469.2076908	total: 22.6s	remaining: 13.9s
1200:	learn: 467.6766924	total: 27.4s	remaining: 9.46s
1400:	learn: 466.3202383	total: 32.1s	remaining: 4.93s
1600:	learn: 465.0871979	total: 37.1s	remaining: 348ms
1615:	learn: 465.0102910	total: 37.5s	remaining: 0us
Saved predictions/catboost_lightgbm_blend_submission.csv


,Id,Predicted
0,0,2808.772227
1,1,3742.268273
2,2,3149.066004
3,3,3140.455298
4,4,3290.601590
